In [1]:

# Instalar Gurobi (si Colab no lo tiene)
!pip -q install gurobipy





[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import gurobipy as gp
from gurobipy import GRB

m = gp.Model("Prueba")

x = m.addVar(vtype=GRB.BINARY, name="x")
y = m.addVar(vtype=GRB.BINARY, name="y")

m.setObjective(x + 2*y, GRB.MAXIMIZE)

m.addConstr(x + y <= 1)

m.optimize()

print("\nSolución:")
for v in m.getVars():
    print(f"{v.VarName} = {v.X}")

print("Objetivo =", m.ObjVal)

Restricted license - for non-production use only - expires 2027-11-29
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 1 rows, 2 columns and 2 nonzeros (Max)
Model fingerprint: 0x9080d21e
Model has 2 linear objective coefficients
Variable types: 0 continuous, 2 integer (2 binary)
Coefficient statistics:
  Matrix range     [1e+00, 1e+00]
  Objective range  [1e+00, 2e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 1e+00]



Found heuristic solution: objective 1.0000000
Presolve removed 1 rows and 2 columns
Presolve time: 0.02s
Presolve: All rows and columns removed

Explored 0 nodes (0 simplex iterations) in 0.05 seconds (0.00 work units)
Thread count was 1 (of 8 available processors)

Solution count 2: 2 1 

Optimal solution found (tolerance 1.00e-04)
Best objective 2.000000000000e+00, best bound 2.000000000000e+00, gap 0.0000%

Solución:
x = 0.0
y = 1.0
Objetivo = 2.0


In [5]:
import os

print("GRB_LICENSE_FILE =", os.environ.get("GRB_LICENSE_FILE"))
print("GRB_WLSACCESSID =", os.environ.get("GRB_WLSACCESSID"))
print("GRB_WLSSECRET =", os.environ.get("GRB_WLSSECRET"))
print("GRB_LICENSEID =", os.environ.get("GRB_LICENSEID"))

GRB_LICENSE_FILE = D:\licencia\gurobi.lic
GRB_WLSACCESSID = None
GRB_WLSSECRET = None
GRB_LICENSEID = None


In [6]:

import gurobipy as gp; print(gp.gurobi.version()); m=gp.Model()

(13, 0, 2)


In [7]:
#importaciones
import gurobipy as gp
from gurobipy import GRB

import time
import math

In [ ]:
###############################################################
# CONFIGURACIÓN DEL MODELO
###############################################################

# Número de rondas
rounds = 3

# Número de lanes
# Keccak usa una matriz 5x5
lanes = 5

# Tamaño de palabra
# Cambiar a 8 para la segunda prueba
bits = 4

# Tiempo máximo para Gurobi
TIME_LIMIT = 1500

print("="*60)
print("MODELO MILP PARA KECCAK")
print("="*60)
print(f"Rondas            : {rounds}")
print(f"Lanes             : {lanes} x {lanes}")
print(f"Bits por lane     : {bits}")
print(f"Estado            : {lanes*lanes*bits} bits")
print(f"S-boxes/ronda     : {lanes*bits}")
print(f"Tiempo límite     : {TIME_LIMIT} s")
print("="*60)

MODELO MILP PARA KECCAK
Rondas            : 4
Lanes             : 5 x 5
Bits por lane     : 4
Estado            : 100 bits
S-boxes/ronda     : 20
Tiempo límite     : 1500 s


In [9]:
###############################################################
# ROTACIONES RHO
###############################################################

rotation_offsets = [

    [0,36,3,41,18],

    [1,44,10,45,2],

    [62,6,43,15,61],

    [28,55,25,21,56],

    [27,20,39,8,14]

]

In [10]:
###############################################################
# CREAR EL MODELO MILP
###############################################################

# Elimina el modelo anterior si existe
try:
    del model
except NameError:
    pass

# Crear un modelo nuevo
model = gp.Model("Keccak_MILP")

# Parámetros del solver
model.Params.TimeLimit = TIME_LIMIT

print("Modelo creado correctamente.")
print("ID del modelo:", id(model))

Set parameter TimeLimit to value 1500
Modelo creado correctamente.
ID del modelo: 2228095985248


In [11]:
##############################################################
# VARIABLES DEL ESTADO
##############################################################

S = model.addVars(

    rounds+1,

    lanes,

    lanes,

    bits,

    vtype=GRB.BINARY,

    name="S"

)
# variables de theta
C = model.addVars(

    rounds,

    lanes,

    bits,

    vtype=GRB.BINARY,

    name="C"

)

D = model.addVars(

    rounds,

    lanes,

    bits,

    vtype=GRB.BINARY,

    name="D"

)
#estado despues de theta
B = model.addVars(

    rounds,

    lanes,

    lanes,

    bits,

    vtype=GRB.BINARY,

    name="B"

)
#Entrada a Chi
Chi_input = model.addVars(

    rounds,

    lanes,

    lanes,

    bits,

    vtype=GRB.BINARY,

    name="ChiInput"

)
#Chi activa
Chi_active = model.addVars(

    rounds,

    lanes,

    bits,

    vtype=GRB.BINARY,

    name="ChiActive"

)
#variables AND
AND = model.addVars(

    rounds,

    lanes,

    bits,

    lanes,

    vtype=GRB.BINARY,

    name="AND"

)

In [12]:
###############################################################
# FUNCIONES AUXILIARES
###############################################################

# =============================================================
# Contador para variables temporales del XOR
# =============================================================

temp_counter = 0
XOR_temp = {}


def new_temp_var():
    """
    Crea una nueva variable binaria temporal.
    Se utiliza para encadenar XOR de varios bits.
    """

    global temp_counter

    temp_counter += 1

    XOR_temp[temp_counter] = model.addVar(
        vtype=GRB.BINARY,
        name=f"XOR_temp_{temp_counter}"
    )

    return XOR_temp[temp_counter]


# =============================================================
# XOR DE DOS VARIABLES
# =============================================================

def add_xor(model, a, b, c):
    """
    Modela:

        c = a XOR b

    mediante restricciones lineales.
    """

    model.addConstr(c <= a + b)

    model.addConstr(a <= b + c)

    model.addConstr(b <= a + c)

    model.addConstr(a + b + c <= 2)


# =============================================================
# XOR DE N VARIABLES
# =============================================================

def add_xor_n(model, inputs, output):
    """
    Modela

        output = XOR(inputs)

    usando variables temporales.
    """

    if len(inputs) == 0:

        model.addConstr(output == 0)

        return

    current = inputs[0]

    for i in range(1, len(inputs)):

        temp = new_temp_var()

        add_xor(model, current, inputs[i], temp)

        current = temp

    if len(inputs) > 1:

        model.addConstr(output == current)

    else:

        model.addConstr(output == inputs[0])


# =============================================================
# OR DE N VARIABLES
# =============================================================

def add_or(model, inputs, output):
    """
    Modela

        output = OR(inputs)

    mediante restricciones lineales.
    """

    if len(inputs) == 0:

        model.addConstr(output == 0)

        return

    for inp in inputs:

        model.addConstr(output >= inp)

    model.addConstr(output <= gp.quicksum(inputs))

In [13]:
###############################################################
# RESTRICCIONES INICIALES
###############################################################

# -------------------------------------------------------------
# Evitar la solución trivial (todo el estado en cero)
# Debe existir al menos un bit activo en el estado inicial.
# -------------------------------------------------------------

model.addConstr(

    gp.quicksum(
        S[0, x, y, z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    ) >= 1,

    name="EstadoInicial"

)

# -------------------------------------------------------------
# (Opcional)
# Algunas formulaciones también obligan a que exista al menos
# un bit activo en el estado final.
# -------------------------------------------------------------

model.addConstr(

    gp.quicksum(
        S[rounds, x, y, z]
        for x in range(lanes)
        for y in range(lanes)
        for z in range(bits)
    ) >= 1,

    name="EstadoFinal"

)

print("✓ Restricciones iniciales agregadas.")

✓ Restricciones iniciales agregadas.


In [14]:
###############################################################
# RESTRICCIONES POR RONDA
###############################################################

print("Añadiendo restricciones...")

for r in range(rounds):

    print(f"  Procesando ronda {r}...")

    # =========================================================
    # 1. PARIDAD DE COLUMNAS (THETA)
    # C[r][x][z] = XOR_y S[r][x][y][z]
    # =========================================================

    for x in range(lanes):
        for z in range(bits):

            entradas = [
                S[r, x, y, z]
                for y in range(lanes)
            ]

            add_xor_n(
                model,
                entradas,
                C[r, x, z]
            )

    # =========================================================
    # 2. DIFUSIÓN (THETA)
    # D[x] = C[x-1] XOR C[x+1]
    # =========================================================

    for x in range(lanes):
        for z in range(bits):

            a = C[r, (x - 1) % lanes, z]

            b = C[
                r,
                (x + 1) % lanes,
                (z - 1) % bits
            ]

            add_xor(
                model,
                a,
                b,
                D[r, x, z]
            )

    # =========================================================
    # 3. THETA COMPLETA
    # B = S XOR D
    # =========================================================

    for x in range(lanes):
        for y in range(lanes):
            for z in range(bits):

                add_xor(
                    model,
                    S[r, x, y, z],
                    D[r, x, z],
                    B[r, x, y, z]
                )

    # =========================================================
    # 4. RHO + PI
    # =========================================================

    for x in range(lanes):
        for y in range(lanes):

            x_prime = (x + 3 * y) % lanes
            y_prime = y

            shift = rotation_offsets[x][y] % bits

            for z in range(bits):

                z_prime = (z - shift) % bits

                model.addConstr(

                    Chi_input[
                        r,
                        x_prime,
                        y_prime,
                        z
                    ]

                    ==

                    B[
                        r,
                        x,
                        y,
                        z_prime
                    ]

                )

    # =========================================================
    # 5. CHI
    # =========================================================

    for y in range(lanes):

        for z in range(bits):

            #
            # Una S-box por fila
            #

            for x in range(lanes):

                x1 = (x + 1) % lanes
                x2 = (x + 2) % lanes

                #
                # AND
                #

                model.addConstr(

                    AND[r, y, z, x]

                    <=

                    Chi_input[
                        r,
                        x1,
                        y,
                        z
                    ]

                )

                model.addConstr(

                    AND[r, y, z, x]

                    <=

                    Chi_input[
                        r,
                        x2,
                        y,
                        z
                    ]

                )

                model.addConstr(

                    AND[r, y, z, x]

                    >=

                    Chi_input[
                        r,
                        x1,
                        y,
                        z
                    ]

                    +

                    Chi_input[
                        r,
                        x2,
                        y,
                        z
                    ]

                    - 1

                )

                #
                # XOR
                #

                add_xor(

                    model,

                    Chi_input[
                        r,
                        x,
                        y,
                        z
                    ],

                    AND[
                        r,
                        y,
                        z,
                        x
                    ],

                    S[
                        r + 1,
                        x,
                        y,
                        z
                    ]

                )

            #
            # ¿La S-box está activa?
            #

            entradas = [

                Chi_input[
                    r,
                    x,
                    y,
                    z
                ]

                for x in range(lanes)

            ]

            add_or(

                model,

                entradas,

                Chi_active[
                    r,
                    y,
                    z
                ]

            )

print("✓ Restricciones agregadas.")

Añadiendo restricciones...
  Procesando ronda 0...
  Procesando ronda 1...
  Procesando ronda 2...
  Procesando ronda 3...
✓ Restricciones agregadas.


In [15]:
###############################################################
# FUNCIÓN OBJETIVO
###############################################################

print("\nConstruyendo la función objetivo...")

# Minimizar el número total de S-boxes activas
objective = gp.quicksum(
    Chi_active[r, y, z]
    for r in range(rounds)
    for y in range(lanes)
    for z in range(bits)
)

model.setObjective(objective, GRB.MINIMIZE)

print("✓ Función objetivo creada.")



Construyendo la función objetivo...
✓ Función objetivo creada.


In [16]:
###############################################################
# RESOLVER EL MODELO
###############################################################

print("\nIniciando optimización...")

inicio = time.time()

model.optimize()

fin = time.time()

print("\nTiempo de ejecución: %.2f segundos" % (fin - inicio))


Iniciando optimización...
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (win64 - Windows 11+.0 (26200.2))

CPU model: Intel(R) Core(TM) i7-10510U CPU @ 1.80GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Non-default parameters:
TimeLimit  1500



GurobiError: Model too large for size-limited license; visit https://gurobi.com/unrestricted for more information

In [18]:
import gurobipy as gp

env = gp.Env()
m = gp.Model(env=env)

Restricted license - for non-production use only - expires 2027-11-29
